<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c11/c11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11 - working along

## Glorot/He Weight Initialization

### Performance of Classification problem of c10 - ex 15 with correct weight initialization

In [1]:
# load the dataset
from sklearn.datasets import fetch_covtype

# Replace fetch_openml with the optimized scikit-learn loader
covtype = fetch_covtype(as_frame=True)

# Your existing logic remains exactly the same
X = covtype.data
y = covtype.target

split into train, validation and test set

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    covtype.data, covtype.target, random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size = 0.2, random_state=42
)

In [3]:
import torch

In [4]:
X_train = torch.FloatTensor(X_train.values)
X_valid = torch.FloatTensor(X_valid.values)
X_test = torch.FloatTensor(X_test.values)

y_train = torch.FloatTensor(y_train.values)
y_valid = torch.FloatTensor(y_valid.values)
y_test = torch.FloatTensor(y_test.values)

y_train = (y_train - 1).long()
y_valid = (y_valid - 1).long()
y_test = (y_test - 1).long()

Standardize the data

In [5]:
means = X_train.mean(dim=0,keepdims=True) # dim=0 to take the mean along columns
stds = X_train.std(dim=0,keepdims=True)
X_train_std = (X_train - means)/stds
X_valid_std = (X_valid - means)/stds
X_test_std = (X_test - means)/stds

In [6]:
from torch.utils.data import TensorDataset, DataLoader

In [7]:
train_std_dataset = TensorDataset(X_train_std,y_train)
valid_std_dataset = TensorDataset(X_valid_std, y_valid)
test_std_dataset = TensorDataset(X_test_std, y_test)

In [8]:
train_std_loader = DataLoader(train_std_dataset, batch_size = 32, shuffle=True)
valid_std_loader = DataLoader(valid_std_dataset, batch_size = 32)
test_std_loader = DataLoader(test_std_dataset, batch_size = 32)

In [9]:
import torch.nn as nn

In [10]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [11]:
class MLP_clf(nn.Module):
  def __init__(self,n_inputs,n_classes):
    super().__init__()
    self.stack = nn.Sequential(
        nn.Linear(n_inputs,200),
        nn.ReLU(),
        nn.Linear(200,100),
        nn.ReLU(),
        nn.Linear(100,50),
        nn.ReLU(),
        nn.Linear(50,n_classes)
    )

  def forward(self,X):
    return self.stack(X)

### Define a function that initializes the weight to every instance of the `nn.Linear` class

In [12]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

Initialize the model and its weights using He initialization

In [13]:
model = MLP_clf(n_inputs=54, n_classes=7)
model.apply(use_he_init)
model = model.to(device)

Define the train function

In [18]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.1 MB/s eta 0:00:00


In [19]:
import torchmetrics

In [20]:
def train2(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=7).to(device)
  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      y_pred_class = y_pred.argmax(dim=1)
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred_class,y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)
    # print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}')

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        y_valid_class = y_valid_pred.argmax(dim=1)
        valid_accuracy.update(y_valid_class, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

    print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}, Validation Accuracy: {epoch_valid_accuracy:.4f}')

In [21]:
model = MLP_clf(n_inputs=54, n_classes=7)
model.apply(use_he_init)
model = model.to(device)

learning_rate = 0.001
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
xentropy = nn.CrossEntropyLoss()

In [22]:
train2(model,optimizer,xentropy,train_std_loader,valid_std_loader,10)

Epoch 1/10: Train Loss: 0.7913 Train Accuracy: 0.6822, Validation Accuracy: 0.7270
Epoch 2/10: Train Loss: 0.6380 Train Accuracy: 0.7355, Validation Accuracy: 0.7454
Epoch 3/10: Train Loss: 0.6031 Train Accuracy: 0.7476, Validation Accuracy: 0.7516
Epoch 4/10: Train Loss: 0.5801 Train Accuracy: 0.7550, Validation Accuracy: 0.7593
Epoch 5/10: Train Loss: 0.5627 Train Accuracy: 0.7615, Validation Accuracy: 0.7651
Epoch 6/10: Train Loss: 0.5483 Train Accuracy: 0.7669, Validation Accuracy: 0.7696
Epoch 7/10: Train Loss: 0.5361 Train Accuracy: 0.7712, Validation Accuracy: 0.7738
Epoch 8/10: Train Loss: 0.5252 Train Accuracy: 0.7756, Validation Accuracy: 0.7804
Epoch 9/10: Train Loss: 0.5150 Train Accuracy: 0.7802, Validation Accuracy: 0.7831
Epoch 10/10: Train Loss: 0.5056 Train Accuracy: 0.7840, Validation Accuracy: 0.7865


In [24]:
torch.manual_seed(42)
model = MLP_clf(n_inputs=54, n_classes=7)
model.apply(use_he_init)
model = model.to(device)

xentropy = nn.CrossEntropyLoss()

for lr in [0.2, 0.1, 0.05, 0.001]:
  optimizer = torch.optim.SGD(model.parameters(), lr = lr)
  train2(model,optimizer, xentropy, train_std_loader, valid_std_loader,10)

Epoch 1/10: Train Loss: 0.5432 Train Accuracy: 0.7691, Validation Accuracy: 0.8053
Epoch 2/10: Train Loss: 0.4805 Train Accuracy: 0.8056, Validation Accuracy: 0.8041
Epoch 3/10: Train Loss: 0.4340 Train Accuracy: 0.8209, Validation Accuracy: 0.8289
Epoch 4/10: Train Loss: 0.3888 Train Accuracy: 0.8394, Validation Accuracy: 0.8290
Epoch 5/10: Train Loss: 0.3664 Train Accuracy: 0.8498, Validation Accuracy: 0.8539
Epoch 6/10: Train Loss: 0.3506 Train Accuracy: 0.8565, Validation Accuracy: 0.8605
Epoch 7/10: Train Loss: 0.3399 Train Accuracy: 0.8631, Validation Accuracy: 0.8514
Epoch 8/10: Train Loss: 0.3299 Train Accuracy: 0.8664, Validation Accuracy: 0.8664
Epoch 9/10: Train Loss: 0.3218 Train Accuracy: 0.8702, Validation Accuracy: 0.8720
Epoch 10/10: Train Loss: 0.3157 Train Accuracy: 0.8729, Validation Accuracy: 0.8495
Epoch 1/10: Train Loss: 0.2628 Train Accuracy: 0.8958, Validation Accuracy: 0.8975
Epoch 2/10: Train Loss: 0.2451 Train Accuracy: 0.9014, Validation Accuracy: 0.8984
Epo

### Initialize the weights directly in the model.

In [25]:
class MLP_he_clf(nn.Module):
  def __init__(self,n_inputs,n_classes): # this is what happens when we initialize the model: model = MLP_he_clf(54,7)
    super().__init__()
    self.stack = nn.Sequential(
        nn.Linear(n_inputs,200),
        nn.ReLU(),
        nn.Linear(200,100),
        nn.ReLU(),
        nn.Linear(100,50),
        nn.ReLU(),
        nn.Linear(50,n_classes)
    )

    with torch.no_grad():
      self.apply(self.use_he_init)

  def use_he_init(self, module):
    if isinstance(module, nn.Linear):
      nn.init.kaiming_uniform_(module.weight)
      nn.init.zeros_(module.bias)

  def forward(self,X): # this is what happens when we give data to the model to make predictions: model(X)
    return self.stack(X)

In [27]:
torch.manual_seed(42)
model2 = MLP_he_clf(n_inputs=54,n_classes=7)
model2 = model2.to(device)
xentropy = nn.CrossEntropyLoss()

for lr in [0.2, 0.1, 0.05, 0.001]:
  optimizer = torch.optim.SGD(model2.parameters(), lr = lr)
  train2(model2, optimizer, xentropy, train_std_loader, valid_std_loader,2)

Epoch 1/2: Train Loss: 0.5432 Train Accuracy: 0.7691, Validation Accuracy: 0.8053
Epoch 2/2: Train Loss: 0.4805 Train Accuracy: 0.8056, Validation Accuracy: 0.8041
Epoch 1/2: Train Loss: 0.3823 Train Accuracy: 0.8409, Validation Accuracy: 0.8504
Epoch 2/2: Train Loss: 0.3431 Train Accuracy: 0.8584, Validation Accuracy: 0.8552
Epoch 1/2: Train Loss: 0.2863 Train Accuracy: 0.8826, Validation Accuracy: 0.8813
Epoch 2/2: Train Loss: 0.2724 Train Accuracy: 0.8882, Validation Accuracy: 0.8880
Epoch 1/2: Train Loss: 0.2380 Train Accuracy: 0.9031, Validation Accuracy: 0.9008
Epoch 2/2: Train Loss: 0.2281 Train Accuracy: 0.9078, Validation Accuracy: 0.9026
